# Train Random Forest v3 trên Kaggle — giữ nguyên Transformer

Notebook độc lập, chuẩn bị từ notebook v3 của dự án ngày 07/11/2026.

**Chỉ train RF.** Không chạy PyTorch, không export hoặc ghi đè Transformer.
Chọn `MODE="npz"` nếu còn dataset_windows.npz gốc; nếu không, chọn `MODE="video"` để dựng lại theo mã cũ.
Video dựng lại có thể khác dataset lịch sử do phiên bản thư viện/cache. Kiểm số lượng không chứng minh giống từng byte.

Đính kèm dataset video cũ và dataset tham chiếu riêng gồm: posture_config.json, posture_transformer.onnx, pose_landmarker_full.task từ repo đang dùng.
Sửa các đường dẫn ở ô **Cấu hình**. CPU đủ cho RF; GPU không cần.
Kết quả cuối: ZIP chứa RF + metadata + dependency versions + dataset + các file kiểm kê. Chưa đánh giá cảnh báo video người mới.

## 1. Cấu hình — sửa ô này trước khi Run All

In [ ]:
from pathlib import Path
import os, glob, re, json, hashlib, shutil, sys, subprocess
from datetime import datetime, timezone

MODE = "video"  # "npz" nếu còn dataset_windows.npz của lần train cũ
DATA_DIR = "/kaggle/input/datasets/vphcvit12a1/datakl/Video_TrainAI_KhoaLuan"
ASSET_DIR = "/kaggle/input/posture-v3-reference"  # sửa đúng thư mục đã Add Input
DATASET_NPZ = "/kaggle/input/posture-v3-windows/dataset_windows.npz"  # chỉ dùng ở MODE=npz
RUN_LOSO = True  # chỉ LOSO của RF; không train lại Transformer
WORKERS = 2     # trích video: 2 process để tránh dùng hết RAM

REFERENCE_HASHES = {
    "posture_config.json":  "03d344a538b488bf4fbd5400b669a15b006d7f90a3323393efd2eda81f39ddea",
    "posture_transformer.onnx":  "f3432a3626d80fdc751be85298a5004abd6d36eb02ebe9b16e588bacd131bb20",
    "pose_landmarker_full.task":  "4eaa5eb7a98365221087693fcc286334cf0858e2eb6e15b506aa4a7ecdcec4ad"
}

def digest(path):
    h = hashlib.sha256()
    with open(path, 'rb') as f:
        for block in iter(lambda: f.read(1024 * 1024), b''):
            h.update(block)
    return h.hexdigest()

if MODE not in ("video", "npz"):
    raise ValueError('MODE must be video or npz')
if not Path(ASSET_DIR).is_dir():
    raise FileNotFoundError(f'Sửa ASSET_DIR. Các thư mục đang có: {list(Path("/kaggle/input").iterdir())}')

assets = {}
for name, expected in REFERENCE_HASHES.items():
    found = list(Path(ASSET_DIR).rglob(name))
    if len(found) != 1:
        raise ValueError(f'Cần đúng một {name} dưới ASSET_DIR, tìm thấy {len(found)}')
    if digest(found[0]) != expected:
        raise ValueError(f'{name} khác bản tham chiếu. Upload đúng file từ repo; không tự bỏ kiểm hash.')
    assets[name] = found[0]

REF_CONFIG = assets['posture_config.json']
TASK = str(assets['pose_landmarker_full.task'])
WORK = '/kaggle/working'
os.chdir(WORK)
RUN_DIR = Path(WORK) / ('rf_run_' + datetime.now(timezone.utc).strftime('%Y%m%dT%H%M%S%fZ'))
RUN_DIR.mkdir(exist_ok=False)
LM_DIR = str(RUN_DIR / 'lm')
Path(LM_DIR).mkdir()
DATASET_PATH = Path(DATASET_NPZ) if MODE == 'npz' else RUN_DIR / 'dataset_windows.npz'
if MODE == 'npz' and not DATASET_PATH.is_file():
    raise FileNotFoundError('Sửa DATASET_NPZ hoặc chọn MODE=video')
print('Run:', RUN_DIR, '\nMode:', MODE, '\nReference:', REF_CONFIG)

## 2. Thư viện

Dùng các thư viện có sẵn trong Kaggle; chỉ cài gói còn thiếu. Nếu phải cài, bật Internet cho notebook rồi chạy lại từ ô cấu hình sau khi restart session nếu Kaggle yêu cầu.
Môi trường sẽ được lưu trong output. Dựng lại video với MediaPipe khác phiên bản lịch sử có thể tạo dữ liệu khác; ưu tiên NPZ gốc nếu còn.

In [ ]:
import importlib.util
required = {'numpy':'numpy', 'pandas':'pandas', 'sklearn':'scikit-learn', 'joblib':'joblib'}
if MODE == 'video':
    required.update({'mediapipe':'mediapipe', 'cv2':'opencv-python'})
missing = [package for module, package in required.items() if importlib.util.find_spec(module) is None]
if missing:
    subprocess.run([sys.executable, '-m', 'pip', 'install', *missing], check=True)
import numpy as np, pandas as pd, sklearn, joblib
print('numpy', np.__version__, '| sklearn', sklearn.__version__, '| joblib', joblib.__version__)

## 3. Đặc trưng — nguyên mã từ notebook v3, không chỉnh ngưỡng

In [ ]:
%%writefile /kaggle/working/posture_features.py
"""
Đặc trưng tư thế + RULA nhóm B + công thức lệch vai — DÙNG CHUNG cho lúc train và lúc chạy realtime.

- Cam hông 90° (train + chạy thật): góc cổ / thân / cúi đầu -> RULA nhóm B (teacher) -> Transformer (student).
- Cam laptop chính diện (chỉ lúc chạy thật): công thức lệch vai, KHÔNG train.

Đầu vào mỗi khung: 33 landmark MediaPipe Pose 2D (toạ độ pixel x, y, visibility),
world landmark 3D chỉ dùng để chọn phía cơ thể gần camera.
"""
import math
import numpy as np

# chỉ số MediaPipe Pose: (trái, phải)
IDX = {"ear": (7, 8), "eye": (2, 5), "shoulder": (11, 12), "hip": (23, 24)}
NOSE = 0

FEATURES = ["neck", "trunk", "head_pitch", "vis_ear", "vis_shoulder", "vis_hip"]
CALIBRATED = ["neck", "trunk", "head_pitch"]      # trừ giá trị lúc ngồi chuẩn của từng người
ANGLE_SCALE = 30.0                                 # chia góc cho 30° trước khi vào model

# ================================================================ CAM HÔNG 90°
def choose_side(lm_seq, world_seq=None):
    """0 = trái, 1 = phải. Ưu tiên world z (nhỏ = gần camera), dự phòng bằng visibility."""
    if world_seq is not None and np.isfinite(world_seq).any():
        zl = np.nanmedian(world_seq[:, 11, 2] + world_seq[:, 23, 2])
        zr = np.nanmedian(world_seq[:, 12, 2] + world_seq[:, 24, 2])
        if np.isfinite(zl) and np.isfinite(zr):
            return 0 if zl < zr else 1
    vl = np.nanmean(lm_seq[:, [7, 11, 23], 2]); vr = np.nanmean(lm_seq[:, [8, 12, 24], 2])
    return 0 if vl >= vr else 1


def facing_sign(lm_seq, side):
    """+1 nếu người quay mặt sang phải khung hình, -1 nếu sang trái (mũi so với tai)."""
    d = np.nanmedian(lm_seq[:, NOSE, 0] - lm_seq[:, IDX["ear"][side], 0])
    return 1.0 if (np.isfinite(d) and d > 0) else -1.0


def _ang(v, f):
    """Góc (độ) của vector v so với phương thẳng đứng hướng LÊN, dương khi nghiêng về phía mặt."""
    return math.degrees(math.atan2(v[0] * f, -v[1]))


def frame_angles(p, side, f):
    """p: (33,3) pixel x,y,vis. Trả dict góc (độ) + visibility, None nếu thiếu tai/vai/hông."""
    g = lambda k: p[IDX[k][side], :2]
    ear, eye, sh, hip = g("ear"), g("eye"), g("shoulder"), g("hip")
    if not np.isfinite(np.concatenate([ear, sh, hip])).all():
        return None
    trunk = _ang(sh - hip, f)                       # thân so với phương đứng (+ = gập ra trước, − = ngả sau)
    neck = _ang(ear - sh, f) - trunk                # cổ so với đường thân (+ = gập)
    # cúi đầu: đường tai -> mắt so với phương ngang (+ = mắt thấp hơn tai = cúi)
    head_pitch = math.degrees(math.atan2(eye[1] - ear[1], (eye[0] - ear[0]) * f)) if np.isfinite(eye).all() else np.nan
    vis = lambda k: float(p[IDX[k][side], 2])
    return dict(neck=neck, trunk=trunk, head_pitch=head_pitch,
                vis_ear=vis("ear"), vis_shoulder=vis("shoulder"), vis_hip=vis("hip"))


def sequence_angles(lm_seq, side, f):
    """(T,33,3) -> (T, len(FEATURES)) góc thô (chưa hiệu chỉnh), NaN nếu thiếu."""
    out = np.full((len(lm_seq), len(FEATURES)), np.nan, np.float32)
    for t, p in enumerate(lm_seq):
        if not np.isfinite(p).any():
            continue
        a = frame_angles(p, side, f)
        if a is not None:
            out[t] = [a[k] for k in FEATURES]
    return out


def calibration_baseline(raw):
    """raw: (T,F) góc thô lúc ngồi chuẩn -> dict baseline cho các góc cần hiệu chỉnh."""
    return {k: float(np.nanmedian(raw[:, FEATURES.index(k)])) for k in CALIBRATED}


def apply_calibration(raw, base):
    x = raw.copy()
    for k in CALIBRATED:
        x[:, FEATURES.index(k)] -= base[k]
    return x


def to_model_input(cal):
    """Góc đã hiệu chỉnh -> mảng chuẩn hoá cho model (NaN -> 0, visibility giữ nguyên 0..1)."""
    x = cal.copy()
    for i, k in enumerate(FEATURES):
        if not k.startswith("vis_"):
            x[:, i] = x[:, i] / ANGLE_SCALE
    return np.nan_to_num(x, nan=0.0).astype(np.float32)

# ================================================================ RULA NHÓM B (teacher)
# McAtamney & Corlett (1993). Chỉ Cổ – Thân – Chân; bỏ nhóm A, điểm cơ, điểm lực.
# Side cam không thấy xoay / nghiêng sang bên -> các cờ điều chỉnh = 0. Chân mặc định = 1.
TABLE_B = {  # neck -> [T1L1, T1L2, T2L1, T2L2, ..., T6L1, T6L2]
    1: [1, 3, 2, 3, 3, 4, 5, 5, 6, 6, 7, 7], 2: [2, 3, 2, 3, 4, 5, 5, 5, 6, 7, 7, 7],
    3: [3, 3, 3, 4, 4, 5, 5, 6, 6, 7, 7, 7], 4: [5, 5, 5, 6, 6, 7, 7, 7, 7, 7, 8, 8],
    5: [7, 7, 7, 7, 7, 8, 8, 8, 8, 8, 8, 8], 6: [8, 8, 8, 8, 8, 8, 8, 9, 9, 9, 9, 9],
}
BAD_SCORE = 3          # điểm nhóm B >= 3 -> BAD
NEUTRAL_TOL = 5.0      # thân: <= 5° so với lúc hiệu chỉnh vẫn coi là thẳng (1 điểm)
NECK_EXT_TOL = 15.0    # cổ chỉ tính "ngửa" (4 điểm) khi ngửa hơn lúc hiệu chỉnh > 15°
LEAN_BACK_TOL = 15.0   # luật bổ sung ngoài RULA: thân ngả sau > 15° -> BAD (LEAN_BACKWARD)
HEAD_FROM_PITCH = True # góc cổ = max(tai–vai so với thân, cúi mặt tai->mắt): bắt kiểu gù mà cúi mặt


def score_neck(a):
    if a < -NECK_EXT_TOL: return 4
    if a <= 10:           return 1
    if a <= 20:           return 2
    return 3


def score_trunk(a):
    if a <= NEUTRAL_TOL: return 1          # thẳng, hoặc ngả sau có tựa
    if a <= 20:          return 2
    if a <= 60:          return 3
    return 4


def group_b_score(neck, trunk, legs=1):
    """Góc cổ, thân (độ, đã hiệu chỉnh) -> điểm nhóm B (1..9), NaN nếu thiếu."""
    if not (np.isfinite(neck) and np.isfinite(trunk)):
        return np.nan
    return TABLE_B[score_neck(neck)][(score_trunk(trunk) - 1) * 2 + (legs - 1)]


def b_label(neck, trunk):
    """-> 0 = GOOD, 1 = BAD, NaN nếu thiếu."""
    b = group_b_score(neck, trunk)
    if not np.isfinite(b):
        return np.nan
    return int(b >= BAD_SCORE or trunk < -LEAN_BACK_TOL)


def rula_frame_label(cal_row, margin=2.0):
    """cal_row: 1 khung đặc trưng đã hiệu chỉnh. Trả (nhãn 0/1/NaN, ổn_định?).
    'Ổn định' = nhãn không đổi khi xê dịch cổ/thân ±margin độ (khung sát ngưỡng sẽ không ổn định)."""
    n, t, hp = (cal_row[FEATURES.index(k)] for k in ("neck", "trunk", "head_pitch"))
    if HEAD_FROM_PITCH and np.isfinite(hp) and np.isfinite(n):
        n = max(n, hp)
    lab = b_label(n, t)
    if not np.isfinite(lab):
        return np.nan, False
    stable = all(b_label(n + dn, t + dt) == lab for dn in (-margin, margin) for dt in (-margin, margin))
    return int(lab), stable

# ================================================================ CAM LAPTOP: LỆCH VAI (không train)
SHOULDER_TILT_ON = 3.5     # |lệch vai so với lúc hiệu chỉnh| > 3.5° -> lệch (clip P01 nghiêng chủ ý chỉ ~4–5°)
SHOULDER_TILT_OFF = 2.5    # < 2.5° mới coi là hết lệch (trễ, tránh bật/tắt liên tục); ngồi thẳng dao động ~±2°
MIN_SHOULDER_VIS = 0.5     # vai bị che / ra khỏi khung -> bỏ khung


def shoulder_tilt(p):
    """p: (33,3) pixel x,y,vis từ cam CHÍNH DIỆN. Góc (độ) của đường nối 2 vai so với phương ngang.
    Đường vai lấy từ vai bên trái ảnh -> vai bên phải ảnh, nên ảnh lật gương (selfie) vẫn đúng độ lớn.
    Dấu: + = vai bên phải ẢNH thấp hơn. Chỉ dùng |tilt − baseline| để cảnh báo."""
    L, R = p[11], p[12]
    if not (np.isfinite(L).all() and np.isfinite(R).all()) or min(L[2], R[2]) < MIN_SHOULDER_VIS:
        return np.nan
    dx, dy = R[0] - L[0], R[1] - L[1]
    if dx < 0:
        dx, dy = -dx, -dy
    if dx < 1e-6:
        return np.nan
    return math.degrees(math.atan2(dy, dx))


def sequence_shoulder_tilt(lm_seq):
    return np.array([shoulder_tilt(p) if np.isfinite(p).any() else np.nan for p in lm_seq], np.float32)

## 4. Kiểm kê nguồn video (bỏ qua trong chế độ NPZ)

In [ ]:
# Ô này tự khai báo helper để chạy lại sau khi sửa cấu hình mà không thiếu digest.
from pathlib import Path
import hashlib, json, re

def digest(path):
    h = hashlib.sha256()
    with open(path, 'rb') as f:
        for block in iter(lambda: f.read(1024 * 1024), b''):
            h.update(block)
    return h.hexdigest()

if 'MODE' not in globals() or MODE not in ('video', 'npz'):
    raise RuntimeError('Chạy ô 1. Cấu hình thành công trước; MODE phải là video hoặc npz.')
required = ('DATA_DIR', 'RUN_DIR') if MODE == 'video' else ('DATASET_PATH',)
missing = [name for name in required if name not in globals()]
if missing:
    raise RuntimeError(f'Thiếu {missing}. Chạy ô 1. Cấu hình thành công rồi chạy lại ô này.')
if MODE == 'video' and not Path(RUN_DIR).is_dir():
    raise RuntimeError('RUN_DIR không tồn tại. Chạy ô 1. Cấu hình để tạo run mới.')

if MODE == 'video':
    if not Path(DATA_DIR).is_dir():
        raise FileNotFoundError('Sửa DATA_DIR thành thư mục video train cũ đã Add Input')
    allv = sorted(str(p) for p in Path(DATA_DIR).rglob('*') if p.suffix.lower() in ('.mp4', '.mov'))
    # Giữ bộ lọc tên như notebook cũ. Hash được lưu để kiểm kê, không tự đổi bộ train.
    allv = [v for v in allv if '(1)' not in v and re.match(r'P\d+_S\d+_', Path(v).stem)]
    side_videos = [v for v in allv if 'SHOULDER_TILT' not in v]
    front_videos = []  # RF chỉ dùng camera hông
    if not side_videos:
        raise ValueError('Không có video train đúng tên Pxx_Sxx_...')
    if len({Path(v).stem for v in side_videos}) != len(side_videos):
        raise ValueError('Tên video trùng ở nhiều thư mục; cần xử lý tránh cache ghi đè')
    manifest = [{'path': str(Path(v).relative_to(DATA_DIR)), 'sha256': digest(v)} for v in side_videos]
    (RUN_DIR / 'video_manifest.json').write_text(json.dumps(manifest, indent=2), encoding='utf-8')
    if len({r['sha256'] for r in manifest}) != len(manifest):
        print('CẢNH BÁO: có video trùng hash. Giữ nguyên để đối chiếu train lịch sử; cần phân tích riêng.')
    print('Video hông:', len(side_videos), '| People:', sorted({Path(v).stem.split('_')[0] for v in side_videos}))
else:
    print('Dùng NPZ có sẵn:', DATASET_PATH)

## 5. Trích landmark từ video cũ

Giữ công thức lấy mẫu/resize của notebook cũ. Dùng asset đã kiểm hash, không tải model `latest`.
Mỗi run có cache mới, tránh tái sử dụng nhầm cache cũ. Nếu dừng giữa chừng, chạy lại riêng ô này trong cùng session để tiếp tục.

In [ ]:
if MODE == 'video':
    import time
    from concurrent.futures import ProcessPoolExecutor
    TARGET_FPS = 10
    
    def extract(v):
        import cv2, numpy as np, mediapipe as mp
        from mediapipe.tasks import python as mpt
        from mediapipe.tasks.python import vision
        name = Path(v).stem; out = f'{LM_DIR}/{name}.npz'
        if os.path.exists(out):
            return name, 'cache'
        det = vision.PoseLandmarker.create_from_options(vision.PoseLandmarkerOptions(
            base_options=mpt.BaseOptions(model_asset_path=TASK), running_mode=vision.RunningMode.VIDEO, num_poses=1))
        cap = cv2.VideoCapture(v); fps = cap.get(cv2.CAP_PROP_FPS) or 30; step = max(1, round(fps / TARGET_FPS))
        L, W, T = [], [], []; i = -1; w = h = 0
        while cap.grab():
            i += 1
            if i % step: continue
            ok, fr = cap.retrieve()
            if not ok: break
            h, w = fr.shape[:2]
            sm = cv2.resize(fr, (w // 2, h // 2))
            r = det.detect_for_video(mp.Image(image_format=mp.ImageFormat.SRGB, data=cv2.cvtColor(sm, cv2.COLOR_BGR2RGB)),
                                     int(i * 1000 / fps))
            if r.pose_landmarks:
                L.append([[p.x * w, p.y * h, p.visibility] for p in r.pose_landmarks[0]])
                W.append([[p.x, p.y, p.z] for p in r.pose_world_landmarks[0]])
            else:
                L.append(np.full((33, 3), np.nan)); W.append(np.full((33, 3), np.nan))
            T.append(i / fps)
        cap.release(); det.close()
        if not T:
            raise ValueError(f'Video không đọc được hoặc rỗng: {v}')
        np.savez_compressed(out, lm=np.array(L, np.float32).reshape(-1, 33, 3), world=np.array(W, np.float32).reshape(-1, 33, 3),
                            t=np.array(T), fps=fps, wh=np.array([w, h]))
        return name, len(T)
    
    videos = side_videos + front_videos
    t0 = time.time()
    with ProcessPoolExecutor(max_workers=WORKERS) as ex:
        for k, (name, n) in enumerate(ex.map(extract, videos), 1):
            print(f'[{k}/{len(videos)}] {name}: {n} khung  ({time.time()-t0:.0f}s)', flush=True)
else:
    print('Bỏ qua trích landmark: dùng NPZ gốc')


## 6. Dựng dataset_windows.npz — giữ cách tạo dữ liệu train lịch sử

Mã gốc chọn bên/hướng từ cả setup và có GOOD fallback khi thiếu CALIBRATION. Giữ để đối chiếu train cũ, **không dùng cách nhìn cả phiên này cho test video người mới**. Chưa chứng minh đầu vào dựng lại giống lịch sử chỉ vì cùng số mẫu.

In [ ]:
if MODE == 'video':
    # Landmark -> góc -> hiệu chỉnh theo người -> nhãn RULA nhóm B -> lọc -> cửa sổ 2 s cho Transformer.
    import sys; sys.path.insert(0, WORK)
    import json, importlib
    import numpy as np, pandas as pd
    import posture_features; importlib.reload(posture_features)
    from posture_features import (FEATURES, choose_side, facing_sign, sequence_angles, calibration_baseline,
                                  apply_calibration, to_model_input, rula_frame_label)
    
    FPS = 10
    WIN, STRIDE = 20, 5            # 2 s, bước 0,5 s
    MIN_PURE = 0.8                 # >= 80% khung cùng nhãn mới giữ
    MIN_STABLE = 0.6               # >= 60% khung "ổn định" (không sát ngưỡng)
    MIN_VALID = 0.8                # >= 80% khung có đủ điểm
    MIN_VIS_HIP = 0.3              # trung bình visibility hông
    
    files = sorted(f for f in glob.glob(f"{LM_DIR}/*.npz") if "SHOULDER_TILT" not in f)
    meta = []
    for f in files:
        name = os.path.basename(f)[:-4]
        m = re.match(r"(P\d+)_(S\d+)_(.+)", name)
        z = np.load(f)
        setup = "x".join(str(int(v)) for v in z["wh"]) if "wh" in z.files else "na"
        meta.append(dict(file=f, name=name, person=m.group(1), session=m.group(2), clip=m.group(3), setup=setup))
    meta = pd.DataFrame(meta)
    
    X, Y, G, INFO, frames_rows = [], [], [], [], []
    for (person, session, setup), grp in meta.groupby(["person", "session", "setup"]):
        data = {r.name: np.load(r.file) for r in grp.itertuples()}
        all_lm = np.concatenate([d["lm"] for d in data.values()])
        all_w = np.concatenate([d["world"] for d in data.values()])
        side = choose_side(all_lm, all_w)
        f = facing_sign(all_lm, side)
        raw = {n: sequence_angles(d["lm"], side, f) for n, d in data.items()}
        calib = [n for n in raw if "CALIBRATION" in n]
        if calib:
            base = calibration_baseline(np.concatenate([raw[n] for n in calib])); src = "CALIBRATION"
        else:
            good = [raw[n][:10 * FPS] for n in raw if "_GOOD_" in n]
            if not good:
                print(f"{person}_{session} [{setup}]: không có CALIBRATION/GOOD -> bỏ"); continue
            base = calibration_baseline(np.concatenate(good)); src = "GOOD (10 s đầu)"
        print(f"{person}_{session} [{setup}, {len(grp)} clip]: side={'trái' if side == 0 else 'phải'} facing={f:+.0f} baseline({src}) "
              + " ".join(f"{k}={v:.1f}" for k, v in base.items()))
        for n, r in raw.items():
            if "CALIBRATION" in n:
                continue
            cal = apply_calibration(r, base)
            labs = [rula_frame_label(row) for row in cal]
            lab = np.array([l for l, _ in labs], float); stab = np.array([s for _, s in labs], bool)
            valid = np.isfinite(cal[:, :2]).all(1)              # cần cổ + thân
            xin = to_model_input(cal)
            clip = n.split("_", 2)[2]
            for t in range(len(cal)):
                frames_rows.append(dict(person=person, session=session, clip=clip, video=n, t=t / FPS,
                                        **{k: cal[t, i] for i, k in enumerate(FEATURES)}, label=lab[t], stable=stab[t]))
            for s in range(0, len(cal) - WIN + 1, STRIDE):
                sl = slice(s, s + WIN)
                v = valid[sl].mean(); l = lab[sl]; ok = np.isfinite(l)
                info = dict(person=person, session=session, clip=clip, video=n, start=s / FPS)
                if v < MIN_VALID or ok.sum() == 0 or np.nanmean(cal[sl, FEATURES.index("vis_hip")]) < MIN_VIS_HIP:
                    INFO.append({**info, "kept": False, "reason": "thiếu điểm"}); continue
                pbad = np.nanmean(l)
                if MIN_PURE > pbad > 1 - MIN_PURE:
                    INFO.append({**info, "kept": False, "reason": "nhãn lẫn"}); continue
                if stab[sl][ok].mean() < MIN_STABLE:
                    INFO.append({**info, "kept": False, "reason": "sát ngưỡng"}); continue
                y = int(pbad >= 0.5)
                X.append(xin[sl]); Y.append(y); G.append(person)
                INFO.append({**info, "kept": True, "reason": "", "label": y})
    
    X, Y, G = np.stack(X), np.array(Y), np.array(G)
    np.savez_compressed(DATASET_PATH, X=X, Y=Y, G=G)
    info = pd.DataFrame(INFO); info.to_csv(RUN_DIR / "windows_info.csv", index=False)
    pd.DataFrame(frames_rows).to_csv(RUN_DIR / "frames_labeled.csv", index=False)
    print("\nCửa sổ giữ lại:", len(Y), "| BAD:", int(Y.sum()), f"({Y.mean():.0%})")
    print(info.assign(reason=info.reason.replace("", "giữ lại")).groupby("reason").size().rename("số cửa sổ").to_string())
    k = info[info.kept]
    print("\n% BAD theo kịch bản quay (kiểm tra nhanh teacher RULA nhóm B):")
    print(k.assign(clip=k["clip"].str.replace(r"_\d+$", "", regex=True)).groupby("clip").label.agg(["mean", "size"]).round(2).to_string())
    print("\nTheo người:", k.groupby("person").label.agg(["mean", "size"]).round(2).to_dict("index"))
else:
    print('Giữ dataset gốc, không tạo nhãn/cửa sổ mới')


## 7. Code train RF và kiểm hợp đồng dữ liệu

In [ ]:
%%writefile /kaggle/working/common.py
"""Shared v3 feature contract. No training or application side effects."""
import hashlib
import importlib.metadata
import json
import platform
from pathlib import Path

import numpy as np

FEATURES = ['neck', 'trunk', 'head_pitch', 'vis_ear', 'vis_shoulder', 'vis_hip']
WINDOW = 20


def sha256(path):
    h = hashlib.sha256()
    with open(path, 'rb') as f:
        for block in iter(lambda: f.read(1024 * 1024), b''):
            h.update(block)
    return h.hexdigest()


def environment():
    out = {'python': platform.python_version()}
    for name in ('numpy', 'scipy', 'scikit-learn', 'joblib', 'onnxruntime',
                 'mediapipe', 'opencv-python', 'threadpoolctl'):
        try:
            out[name] = importlib.metadata.version(name)
        except importlib.metadata.PackageNotFoundError:
            pass
    return out


def write_json(path, value):
    Path(path).write_text(json.dumps(value, ensure_ascii=False, indent=2,
                                    allow_nan=False), encoding='utf-8')


def win_stats(x):
    x = np.asarray(x, dtype=np.float32)
    if x.ndim != 3 or x.shape[1:] != (WINDOW, len(FEATURES)) or not np.isfinite(x).all():
        raise ValueError('Expected finite normalized X with shape [N,20,6]')
    return np.concatenate([x.mean(1), x.std(1), x.min(1), x.max(1)], axis=1)


def load_dataset(path):
    # The original notebook saves G as a Unicode array, so pickle is unnecessary.
    with np.load(path, allow_pickle=False) as data:
        x, y, g = data['X'], data['Y'], data['G']
    win_stats(x)
    if y.shape != (len(x),) or g.shape != y.shape or len(x) == 0:
        raise ValueError('X/Y/G lengths do not match or dataset is empty')
    if set(np.unique(y)) != {0, 1}:
        raise ValueError('Training labels must contain both 0 and 1')
    if g.dtype.kind not in 'US' or any(not str(v).strip() for v in g):
        raise ValueError('G must contain nonempty string person IDs')
    if np.any(x[:, :, 3:] < 0) or np.any(x[:, :, 3:] > 1):
        raise ValueError('Visibility must be in [0,1]')
    return x.astype(np.float32), y.astype(int), g.astype(str)


def check_reference(x, y, g, cfg):
    if cfg['features'] != FEATURES or cfg['window'] != WINDOW or cfg['angle_scale'] != 30:
        raise ValueError('Reference feature/window/normalization contract is incompatible')
    counts = {str(p): int((g == p).sum()) for p in np.unique(g)}
    expected = {p: int(v['n']) for p, v in cfg['loso_transformer'].items()}
    if len(x) != cfg['n_train_windows'] or counts != expected:
        raise ValueError(f'Dataset differs from reference counts: {counts}; expected {expected}')
    if not np.isclose(y.mean(), cfg['bad_ratio'], atol=1e-7, rtol=0):
        raise ValueError('BAD ratio differs from original model configuration')
    return counts


In [ ]:
%%writefile /kaggle/working/train_rf.py
"""Fit ONLY RF on the old v3 dataset; preserve the deployed Transformer."""
import argparse
import json
from pathlib import Path

import joblib
import numpy as np
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import confusion_matrix

from common import (FEATURES, WINDOW, check_reference, environment, load_dataset,
                    sha256, win_stats, write_json)


def train(dataset, config, out, *, loso=False, jobs=-1):
    cfg = json.loads(Path(config).read_text(encoding='utf-8-sig'))
    x, y, g = load_dataset(dataset)
    counts = check_reference(x, y, g, cfg)
    out = Path(out)
    out.mkdir(parents=True, exist_ok=False)
    features = win_stats(x)
    params = dict(n_estimators=300, class_weight='balanced', random_state=0, n_jobs=jobs)
    fold_results = {}
    if loso:
        for person in sorted(counts):
            mask = g != person
            if set(np.unique(y[mask])) != {0, 1}:
                raise ValueError(f'Train fold {person} does not contain two classes')
            rf = RandomForestClassifier(**params).fit(features[mask], y[mask])
            pred = rf.predict(features[~mask])
            tn, fp, fn, tp = confusion_matrix(y[~mask], pred, labels=[0, 1]).ravel()
            fold_results[person] = dict(n=int((~mask).sum()), tn=int(tn), fp=int(fp),
                                       fn=int(fn), tp=int(tp), accuracy=float((pred == y[~mask]).mean()))
    rf = RandomForestClassifier(**params).fit(features, y)
    model_path = out / 'posture_rf.joblib'
    joblib.dump(rf, model_path, compress=3)
    restored = joblib.load(model_path)  # artifact just written by this process
    np.testing.assert_allclose(rf.predict_proba(features[:32]),
                               restored.predict_proba(features[:32]), atol=0, rtol=0)
    env = environment()
    metadata = dict(format_version=1, features=FEATURES, window=WINDOW, angle_scale=30,
                    stats_order=['mean', 'std_ddof0', 'min', 'max'], parameters=params,
                    people=sorted(counts), counts=counts, n=len(y), bad_count=int(y.sum()),
                    dataset_sha256=sha256(dataset), reference_config_sha256=sha256(config),
                    model_sha256=sha256(model_path), environment=env, loso=fold_results,
                    provenance='Counts and class ratio match reference; not proof of historical byte identity.',
                    intended_test='New people excluded from all original training data')
    write_json(out / 'rf_metadata.json', metadata)
    packages = ('numpy', 'scipy', 'scikit-learn', 'joblib', 'threadpoolctl')
    (out / 'rf-requirements.txt').write_text(''.join(f'{p}=={env[p]}\n' for p in packages), encoding='utf-8')
    print(json.dumps({'output': str(out), 'counts': counts, 'bad': int(y.sum()),
                      'model_sha256': metadata['model_sha256']}, indent=2))
    return metadata


if __name__ == '__main__':
    ap = argparse.ArgumentParser(description=__doc__)
    ap.add_argument('--dataset', required=True, type=Path)
    ap.add_argument('--reference-config', required=True, type=Path)
    ap.add_argument('--out', required=True, type=Path, help='New directory, never overwrite a run')
    ap.add_argument('--loso', action='store_true', help='Optional diagnostic RF LOSO; no Transformer training')
    ap.add_argument('--jobs', type=int, default=-1)
    a = ap.parse_args()
    train(a.dataset, a.reference_config, a.out, loso=a.loso, jobs=a.jobs)


## 8. Kiểm dữ liệu trước train

Phải khớp: 9.600 cửa sổ, P01=2.843/P02=3.521/P03=3.236 và tỷ lệ BAD trong config. Nếu không khớp, notebook dừng: kiểm dataset, phiên bản MediaPipe, asset và cache; **không bỏ kiểm để ép chạy**.
Khớp số lượng chỉ là kiểm nhất quán; hash của dataset lịch sử (nếu còn) mới giúp đối chiếu chính xác file đó.

In [ ]:
sys.path.insert(0, WORK)
import importlib, common, train_rf
importlib.reload(common)
importlib.reload(train_rf)
X, Y, G = common.load_dataset(DATASET_PATH)
cfg = json.loads(REF_CONFIG.read_text(encoding='utf-8-sig'))
print('Counts:', {str(p): int((G == p).sum()) for p in np.unique(G)}, '| BAD:', int(Y.sum()), '| shape:', X.shape)
common.check_reference(X, Y, G, cfg)
# Kiểm thứ tự RF: mean(6) -> std(6) -> min(6) -> max(6), không đảo cột.
probe = np.arange(120, dtype=np.float32).reshape(1,20,6)
stats = common.win_stats(probe)
np.testing.assert_array_equal(stats[0,:6], probe.mean(1)[0])
np.testing.assert_array_equal(stats[0,6:12], probe.std(1)[0])
np.testing.assert_array_equal(stats[0,12:18], probe.min(1)[0])
np.testing.assert_array_equal(stats[0,18:], probe.max(1)[0])
print('Kiểm cấu trúc/thống kê đạt. Dataset SHA256:', common.sha256(DATASET_PATH))

## 9. Train RF — CPU, không train Transformer

RF cuối dùng toàn bộ train cũ để kiểm **người mới** sau này. LOSO tùy chọn ở đây chỉ kiểm RF trên nhãn/cửa sổ cũ; không phải kết quả so sánh cảnh báo ba phương pháp trên người mới.
Chạy lại ô này tạo output mới, không ghi đè lần trước.

In [ ]:
RF_OUT = RUN_DIR / ('rf_' + datetime.now(timezone.utc).strftime('%H%M%S%f'))
metadata = train_rf.train(DATASET_PATH, REF_CONFIG, RF_OUT, loso=RUN_LOSO, jobs=-1)
metadata['input_mode'] = MODE
metadata['reference_assets'] = REFERENCE_HASHES
metadata['source_pipeline'] = 'notebook_v3_1cam90_kaggle.ipynb cells 6/8/10; preprocessing only'
common.write_json(RF_OUT / 'rf_metadata.json', metadata)
if metadata['loso']:
    display(pd.DataFrame.from_dict(metadata['loso'], orient='index'))
print('RF cuối đã lưu:', RF_OUT)

## 10. Đóng gói và tải xuống

Tải ZIP trong Output sau khi Save Version thành công. ZIP không chứa video người tham gia hoặc cache landmark.
Giữ file RF cùng metadata và rf-requirements.txt. Nạp joblib chỉ từ nguồn tin cậy, bằng môi trường tương thích với phiên bản đã ghi.
Model Transformer giữ nguyên; chưa có kết quả cảnh báo thực tế ở bước này.

In [ ]:
EXPORT = RUN_DIR / ('export_' + datetime.now(timezone.utc).strftime('%H%M%S%f'))
shutil.copytree(RF_OUT, EXPORT)
shutil.copy2(DATASET_PATH, EXPORT / 'dataset_windows.npz')
shutil.copy2(REF_CONFIG, EXPORT / 'reference_posture_config.json')
for name in ('posture_features.py', 'common.py', 'train_rf.py'):
    shutil.copy2(Path(WORK) / name, EXPORT / name)
for name in ('video_manifest.json', 'windows_info.csv', 'frames_labeled.csv'):
    if (RUN_DIR / name).exists():
        shutil.copy2(RUN_DIR / name, EXPORT / name)
common.write_json(EXPORT / 'environment.json', common.environment())
assert all(digest(assets[name]) == expected for name, expected in REFERENCE_HASHES.items())
zip_path = shutil.make_archive(str(EXPORT), 'zip', EXPORT)
print('TẢI FILE:', zip_path)
print('Transformer/config/pose asset tham chiếu vẫn nguyên hash.')
from IPython.display import FileLink, display
display(FileLink(str(Path(zip_path).relative_to(WORK))))